# 🏋️ 헬스 기록 분석기 v0.1

아이폰 메모에 적던 운동 기록을 **그대로 붙여넣으면** → 표로 바꾸고 → 성장을 계산하고 → **다음 운동의 무게**를 제안해요.

**사용법**
1. 아래 ①의 `MEMO` 칸에 내 메모를 붙여넣기
2. 메뉴 `런타임(Runtime) → 모두 실행(Run all)`
3. 맨 아래 🔧 에러 연습 셀에서 빨간 에러가 나며 멈추는 건 **정상**이에요 (일부러 만든 에러)

**메모 쓰는 법** — 4분할(가슴 · 등 · 어깨 · 하체)
```
가슴                               ← 분할 줄: 이 아래가 한 번의 운동 (날짜는 안 써도 됨)
◦ Bench press: 85*8*4              ← 무게*횟수*세트수
◦ DB press: 44*16,16,16,14         ← 무게*횟수 뒤에 횟수만 이어 쓰기 = 같은 무게로 한 세트씩
◦ Bench lateral raise: 14*22 12*22*3   ← 세트마다 무게가 다르면 띄어 쓰기
◦ Pull up: 13,12,11,11             ← 맨몸 운동은 횟수만
◦ Dips: 12*3                       ← 맨몸 운동의 횟수*세트수 (②에서 증량이 0인 종목)
✓ Chest fly: 60*15,15,13           ← ✓ = 그날 안 한 운동 → 자동으로 빠짐
```
- 같은 분할이 메모에 **또 나오면 다음 회차**로 쳐요. 위에서 아래로 = 예전 → 최근.
- `◦`, `:`는 자동으로 지워요. `*` 대신 `x`, `×`도 돼요.
- 세트 뒤에 쓴 글자(예: `오늘 쉬웠음`)는 무시해요.
- 아직 직접 바꿔 줘야 하는 것: 슈퍼세트 `A/B`는 두 줄로 나누기, `바 제외25*15`는 `25*15`로 쓰기

**데이터 흐름**
```
메모(글자) → parse_memo() → rows(dict 목록) → df(표: 한 줄 = 한 세트) → ⑤회차별 요약 ⑥성장 ⑦사이클 세트 ⑧다음 무게
```

## ① 메모 붙여넣기
`"""` 와 `"""` 사이에 메모를 붙여넣어요.

`"""`(따옴표 3개)는 **여러 줄짜리 글자**의 시작과 끝 표시예요.

In [ ]:
MEMO = """
가슴
◦ Free IBP: 80*9,9
◦ Pull up: 13,12,11,11
◦ Leg raise: 35,25,25,25
◦ Dips: 12*3
◦ BB curl: 40*9*3
◦ Hammer curl: 24*12
◦ DB press: 44*16,16,16,14
✓ Incline Bench press: 80*12,11,9 75*10
✓ Chest fly: 60*15,15,13
✓ SLL/BLL: 12*15/15*3
✓ Pull up/Dips: 10/10

등
◦ Straight arm pull down: 55*13,13,13
◦ Deadlift: 130*8,8,8
◦ Wide pull up: 11,10,9
◦ SLL: 16*22*4
◦ Long pull: 90*16,17 95*12*2
◦ Cable push down: 35*10*2,10
◦ Overhead extension: 35*12*2,10
◦ HLR: 36,30,25
✓ Pull up: 12,11
✓ BLL/SLL: 10*15/20, 16/21

어깨
◦ Smith behind neck press: 65*9,9,9,8
◦ DB shoulder press: 44*16,15,15,13
◦ Peck deck fly: 20*19,19,17,17
◦ Smith SLL: 7*17*3
◦ Bench lateral raise: 14*22 12*22*3
◦ Dumbbell curl: 24*10
✓ Pull up: 8,8,8,8
✓ Dips: 10,10,10,9

하체
◦ Squat: 115*10,10,8 110*9
◦ Kettle bell wide squat: 24*20*3
◦ DB row: 36*16,14,14
◦ EZ bar curl: 25*15 31*10,10
◦ Hammer curl: 24*12*3
✓ Wide Squat: 110*10,10,10,8
✓ Incline D.B curl: 24*12*3
"""

## ② 설정: 분할 · 종목 (부위 · 반복 범위 · 증량 무게)
**여기만 고치면** 아래 결과가 전부 바뀌어요.
- `DAYS`: 메모에서 **이 이름으로 시작하는 줄**이 한 번의 운동의 시작이에요.
- **범위** `(최소, 최대)`: 모든 세트가 **최대**에 도달하면 증량. 예) `(8, 10)` 60kg → 8/8/8 … 10/10/10 달성 → 65kg
- **증량**: 한 번에 올릴 무게(kg). **맨몸 운동은 0** → 메모의 `12*3`을 "12회 × 3세트"로 읽어요. (중량 풀업처럼 무게를 달면 0이 아닌 값으로)
- 설정에 없는 종목은 `DEFAULT` 값을 써요.

| 기준 | 값 | 근거 |
|---|---|---|
| 부위별 주간 세트 하한 | 10세트 | Schoenfeld 외 2017 메타분석: 주 10세트 이상이 그보다 적을 때보다 근비대가 컸다 |
| 부위별 주간 세트 상한 | 상한없음 | |
| 증량 규칙 | 목표달성 → 증량 | ACSM 2009 권고: 목표 횟수를 1-2회 넘기는 날이 2회 연속이면 2~10% 증량 |
| 목표 반복 범위 | 종목마다 (`EXERCISES`의 범위) | 실패 근처까지 하면 넓은 범위(저중량~고중량)에서 근비대는 비슷 (Schoenfeld 외 2017) |

In [ ]:
import pandas as pd

DAYS = ["가슴", "등", "어깨", "하체"]   # 4분할

# 쓸 수 있는 범위 예: (4, 6) (6, 8) (8, 10) (10, 12) (12, 15) (15, 20)
EXERCISES = {
    # 가슴
    "Bench press":             {"부위": "가슴", "범위": (6, 8),   "증량": 5},
    "Free IBP":                {"부위": "가슴", "범위": (8, 10),  "증량": 5},
    "DB press":                {"부위": "가슴", "범위": (12, 15), "증량": 2},
    # 등
    "Pull up":                 {"부위": "등",   "범위": (8, 12),  "증량": 0},
    "Wide pull up":            {"부위": "등",   "범위": (8, 12),  "증량": 0},
    "Straight arm pull down":  {"부위": "등",   "범위": (12, 15), "증량": 5},
    "Deadlift":                {"부위": "등",   "범위": (6, 8),   "증량": 5},
    "Long pull":               {"부위": "등",   "범위": (12, 15), "증량": 5},
    "DB row":                  {"부위": "등",   "범위": (12, 15), "증량": 2},
    # 어깨
    "Smith behind neck press": {"부위": "어깨", "범위": (8, 10),  "증량": 5},
    "DB shoulder press":       {"부위": "어깨", "범위": (12, 15), "증량": 2},
    "Peck deck fly":           {"부위": "어깨", "범위": (15, 20), "증량": 5},
    "Smith SLL":               {"부위": "어깨", "범위": (15, 20), "증량": 2.5},
    "Bench lateral raise":     {"부위": "어깨", "범위": (15, 20), "증량": 2},
    "SLL":                     {"부위": "어깨", "범위": (15, 20), "증량": 2},
    # 하체
    "Squat":                   {"부위": "하체", "범위": (8, 10),  "증량": 5},
    "Kettle bell wide squat":  {"부위": "하체", "범위": (15, 20), "증량": 4},
    # 팔
    "Dumbbell curl":           {"부위": "이두", "범위": (10, 12), "증량": 2},
    "BB curl":                 {"부위": "이두", "범위": (8, 10),  "증량": 5},
    "Hammer curl":             {"부위": "이두", "범위": (10, 12), "증량": 2},
    "EZ bar curl":             {"부위": "이두", "범위": (10, 12), "증량": 5},
    "Dips":                    {"부위": "삼두", "범위": (10, 15), "증량": 0},
    "Cable push down":         {"부위": "삼두", "범위": (10, 12), "증량": 5},
    "Overhead extension":      {"부위": "삼두", "범위": (10, 12), "증량": 5},
    # 복근
    "Leg raise":               {"부위": "복근", "범위": (20, 30), "증량": 0},
    "HLR":                     {"부위": "복근", "범위": (20, 30), "증량": 0},
}

# 같은 종목의 다른 이름 → EXERCISES에 있는 이름
ALIASES = {
    "IBP": "Free IBP",
}

DEFAULT = {"부위": "미분류", "범위": (8, 12), "증량": 5}   # 설정에 없는 종목

## ③ 메모 → 세트 목록 (`parse_memo`)
- **들어오는 것**: `MEMO` (여러 줄짜리 글자)
- **나가는 것**: `rows` (세트 하나 = dict 하나인 목록), `skipped` (이해 못 한 줄 목록)
- **고칠 곳**: 세트 쓰는 법이 바뀌면 `parse_set`, 분할 이름은 ②의 `DAYS`, 별명은 ②의 `ALIASES`

메모를 한 줄씩 읽으면서 판단해요: **빈 줄** / **✓ 줄(안 한 운동)** / **분할 줄** / **운동 줄**.
분할 줄을 만나면 회차를 1 올리고, 그 아래 운동 줄들은 모두 그 회차로 기록해요.

In [ ]:
def parse_set(token):
    """'60x10' → (60.0, 10, 1),  '60*10*3' → (60.0, 10, 3).  세트 모양이 아니면 None."""
    parts = token.lower().replace("*", "x").replace("×", "x").split("x")
    if len(parts) not in (2, 3):
        return None
    if not parts[0].replace(".", "", 1).isdigit():     # 무게: 60, 62.5 처럼 소수점 1개까지
        return None
    if not all(p.isdigit() for p in parts[1:]):        # 횟수·세트수: 정수만
        return None
    weight = float(parts[0])
    reps = int(parts[1])
    count = int(parts[2]) if len(parts) == 3 else 1
    return weight, reps, count


def is_bodyweight(name_words):
    """②의 설정에서 증량이 0인 종목(맨몸 운동)이면 True."""
    name = " ".join(name_words)
    name = ALIASES.get(name, name)
    return EXERCISES.get(name, DEFAULT)["증량"] == 0


def parse_memo(memo):
    rows = []             # 결과: 세트 하나 = dict 하나
    skipped = []          # 이해 못 한 줄
    session = 0           # 몇 번째 운동인지 (메모 위에서부터 1, 2, 3 …)
    current_day = None    # 지금 읽고 있는 분할 (가슴/등/어깨/하체)

    for line in memo.splitlines():
        tokens = line.replace(",", " ").replace(":", " ").split()
        if not tokens:                        # 빈 줄
            continue
        if tokens[0] == "✓":                  # ✓ = 그날 안 한 운동
            continue
        if tokens[0] in ("◦", "•", "-"):      # 글머리 기호는 지움
            tokens = tokens[1:]

        if tokens[0] in DAYS:                 # 분할 줄 → 새 운동 시작
            session += 1
            current_day = tokens[0]
            continue

        name_words, sets = [], []             # 운동 줄
        for token in tokens:
            s = parse_set(token)
            if s:
                if is_bodyweight(name_words) and s[0] > 0:   # 맨몸 운동의 '12*3' = 12회 × 3세트
                    s = (0.0, int(s[0]), s[1])
                sets.append(s)
            elif token.isdigit() and sets:        # 60*12,12,10 의 '12', '10' → 앞 세트와 같은 무게
                sets.append((sets[-1][0], int(token), 1))
            elif token.isdigit() and name_words:  # Pull up 13,12 → 맨몸(무게 0)
                sets.append((0.0, int(token), 1))
            elif not sets:                        # 첫 세트가 나오기 전 글자 = 종목 이름
                name_words.append(token)

        if current_day is None or not sets or not name_words:
            skipped.append(line)
            continue

        name = " ".join(name_words)
        name = ALIASES.get(name, name)        # 별명이면 정식 이름으로

        set_no = 0
        for weight, reps, count in sets:
            for _ in range(count):            # 60*10*3 → 3번 반복해서 3줄
                set_no += 1
                rows.append({"회차": session, "분할": current_day, "종목": name,
                             "세트": set_no, "무게": weight, "횟수": reps})
    return rows, skipped


rows, skipped = parse_memo(MEMO)
print(f"세트 {len(rows)}개를 읽었어요. 첫 번째:", rows[0])
for line in skipped:
    print("⚠️ 이해 못 한 줄 (건너뜀):", line.strip())

## ④ 표 만들기 + 계산 열 추가
- **들어오는 것**: `rows` (dict 목록)
- **나가는 것**: `df` (표: 한 줄 = 한 세트) + 부위 · 볼륨 · e1RM 열
- **고칠 곳**: 새로 계산하고 싶은 열이 있으면 여기에 한 줄

열 3개를 더해요.
- **부위**: ②의 `EXERCISES`에서 찾기. 없으면 멈추지 않고 `"미분류"`
- **볼륨** = 무게 × 횟수 (맨몸 운동은 0)
- **e1RM**(추정 1회 최대 무게) = 무게 × (1 + 횟수/30) — Epley 공식. *무게와 횟수가 다른 세트끼리 비교하려고* 써요.

In [ ]:
def setting_of(name):
    """종목 설정을 돌려준다. 설정에 없으면 DEFAULT."""
    return EXERCISES.get(name, DEFAULT)


df = pd.DataFrame(rows)
df["부위"] = [setting_of(name)["부위"] for name in df["종목"]]
df["볼륨"] = df["무게"] * df["횟수"]
df["e1RM"] = (df["무게"] * (1 + df["횟수"] / 30)).round(1)
df = df[["회차", "분할", "부위", "종목", "세트", "무게", "횟수", "볼륨", "e1RM"]]   # 열 순서 정리

for name in sorted(df.loc[df["부위"] == "미분류", "종목"].unique()):
    print(f'⚠️ 설정에 없는 종목: "{name}" → ②의 EXERCISES에 추가하세요 (지금은 기본값 사용)')

df.tail(10)

## ⑤ 회차별 요약: 종목 수 · 총 세트 · 총 볼륨
- **들어오는 것**: ④의 `df` (한 줄 = 세트)
- **나가는 것**: 한 줄 = 운동 한 번(회차)
- **고칠 곳**: 보고 싶은 값이 있으면 `.agg( )` 안에 `새이름=(열, 계산)` 한 줄

In [ ]:
summary = df.groupby(["회차", "분할"]).agg(
    종목수=("종목", "nunique"),
    총세트=("세트", "count"),
    총볼륨=("볼륨", "sum"),
)
summary

## ⑥ 종목별 성장 (같은 분할끼리 비교)
- **들어오는 것**: ④의 `df` (한 줄 = 세트)
- **나가는 것**: 한 줄 = (분할, 종목). 그 분할에서 **첫 회차 vs 최근 회차**
- **고칠 곳**: 2단계 `.agg( )` 안에 한 줄

가슴 날의 Hammer curl은 가슴 날끼리, 하체 날의 Hammer curl은 하체 날끼리 비교해요.
같은 분할을 **두 번 이상** 기록해야 변화가 보여요 (지금은 회차가 하나뿐이라 전부 0%).

In [ ]:
# 1단계: (분할, 종목, 회차)마다 '그날 최고 e1RM'과 '그날 볼륨 합계'
per_session = df.groupby(["분할", "부위", "종목", "회차"]).agg(
    e1RM=("e1RM", "max"),
    볼륨=("볼륨", "sum"),
).reset_index()

# 2단계: (분할, 종목)마다 첫 회차 vs 최근 회차
growth = per_session.groupby(["분할", "부위", "종목"]).agg(
    기록횟수=("회차", "count"),
    첫_e1RM=("e1RM", "first"),
    최근_e1RM=("e1RM", "last"),
    첫_볼륨=("볼륨", "first"),
    최근_볼륨=("볼륨", "last"),
)
growth["e1RM 변화(%)"] = ((growth["최근_e1RM"] / growth["첫_e1RM"] - 1) * 100).round(1)
growth["볼륨 변화(%)"] = ((growth["최근_볼륨"] / growth["첫_볼륨"] - 1) * 100).round(1)
growth

## ⑦ 한 사이클의 부위별 세트
- **들어오는 것**: ④의 `df`
- **나가는 것**: 분할마다 **가장 최근 회차 1번씩**(가슴·등·어깨·하체 한 바퀴)을 모아서 센 부위별 세트 수

날짜 대신 "4분할 한 바퀴"를 일주일처럼 봐요. 다른 부위 날에 한 운동(예: 어깨 날의 Dumbbell curl)도 그 부위로 세요.

In [ ]:
latest = df.groupby("분할")["회차"].max()        # 분할마다 가장 최근 회차 번호
cycle = df[df["회차"].isin(latest)]              # 그 회차들의 세트만 남기기

cycle.groupby("부위").size().to_frame("세트 수")  # 줄 수 = 세트 수

## ⑧ 다음 무게
- **들어오는 것**: ④의 `df` + ②의 설정
- **나가는 것**: (분할, 종목)마다 다음 목표
- **고칠 곳**: 규칙을 바꾸려면 `next_weight`

같은 분할의 **가장 최근 회차**를 보고 정해요. **본 세트** = 그날 가장 무거운 무게로 한 세트.

| 조건 | 다음 목표 |
|---|---|
| 본 세트 뒤에 무게를 낮춘 세트(백오프)가 있음 (예: 14×22 → 12×22×3) | 같은 무게로 **본 세트 1개 더** (안 되면 낮춘 무게로) |
| 맨몸 운동 (무게 0) | 세트마다 1회씩 더 |
| 모든 본 세트가 범위 **최대** 이상 | + 증량 무게 |
| 가장 많이 한 세트도 범위 **최소** 미만 | − 증량 무게 |
| 그 외 | 유지, 최대 횟수까지 채우기 |

In [ ]:
def next_weight(weight, reps, low, high, step):
    """최근 본 세트(무게, [횟수들])와 설정(범위, 증량)으로 다음 무게와 이유를 돌려준다."""
    if weight == 0:                                   # 맨몸 운동은 무게 대신 횟수를 늘린다
        return 0, "맨몸 → 같은 세트 수로 세트마다 1회씩 더"
    if min(reps) >= high:
        return weight + step, f"모든 세트 {high}회 달성 → +{step:g}kg"
    if max(reps) < low:
        return max(weight - step, 0), f"최고 {max(reps)}회 < {low}회 → -{step:g}kg"
    return weight, f"같은 무게로 모든 세트 {high}회까지"


plans = []
for (day, exercise), g in df.groupby(["분할", "종목"]):
    last = g[g["회차"] == g["회차"].max()]                   # 이 분할에서 가장 최근 회차
    weights = last["무게"].tolist()
    top = max(weights)                                       # 그날 가장 무거운 무게 = 본 세트
    reps = last.loc[last["무게"] == top, "횟수"].tolist()
    after_top = weights[weights.index(top):]                 # 첫 본 세트부터 끝까지

    setting = setting_of(exercise)
    low, high = setting["범위"]
    if min(after_top) < top:                                 # 본 세트 뒤에 무게를 낮췄으면 (백오프)
        new_weight = top
        reason = f"백오프 있음 → {top:g}kg로 {len(reps) + 1}세트 도전 (안 되면 {min(after_top):g}kg)"
    else:
        new_weight, reason = next_weight(top, reps, low, high, setting["증량"])

    plans.append({
        "분할": day,
        "부위": setting["부위"],
        "종목": exercise,
        "범위": f"{low}~{high}회",
        "지난번": f"{top:g}kg × " + "/".join(str(r) for r in reps),
        "다음 무게": f"{new_weight:g}kg",
        "이유": reason,
    })

pd.DataFrame(plans)    # groupby가 가나다순으로 묶어서 가슴 → 등 → 어깨 → 하체 순서가 돼요

## ⑨ (선택) 표를 CSV 파일로 저장
나중에 모바일 앱을 만들 때 이 표 모양(열 이름)이 그대로 설계도가 돼요.

In [ ]:
df.to_csv("gym_log.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: 엑셀로 열어도 한글이 안 깨져요
print("저장했어요: 왼쪽 📁(파일) 탭 → gym_log.csv → ⋮ → 다운로드")
print("※ Colab 런타임이 끝나면 이 파일은 사라져요. 원본 기록은 아이폰 메모에 그대로 있어요.")

## 🔧 에러 연습 (일부러 에러를 내는 셀)
실행하기 **전에** 무슨 에러가 날지 한 줄로 예상해 보세요.

에러 메시지(Traceback) 읽는 순서:
1. **맨 마지막 줄**: `에러 종류: 설명` (예: `KeyError: ...`)
2. 위로 올라가며 **`---->` 화살표가 가리키는 내 코드 줄** 찾기
3. AI에게 전달: *"`EXERCISES[...]` 줄에서 `KeyError: 'Incline DB press'`가 났어. 설정에 없는 종목을 찾았을 때야."*

In [ ]:
# 연습 1: 사전에 없는 이름을 [ ]로 찾으면? → setting_of()에서 EXERCISES[...] 대신 .get()을 쓴 이유
EXERCISES["Incline DB press"]

In [ ]:
# 연습 2: 글자가 섞인 숫자를 int()로 바꾸면? → parse_set이 먼저 .isdigit()으로 확인하는 이유
int("12회")

## ✅ 점검 질문
1. 메모의 `◦ Pull up: 13,12,11,11` 한 줄은 표에서 몇 줄이 되고, 무게는 몇일까? ③의 어느 줄 때문일까?
2. `Dips: 12*3`은 "12회 × 3세트"로, `Dumbbell curl: 24*10`은 "24kg × 10회"로 읽혀. 둘을 어떻게 구분할까? (힌트: ②의 설정과 ③의 `is_bodyweight`)
3. Bench lateral raise(`14*22 12*22*3`)의 다음 목표가 "14kg로 2세트 도전"이 되는 건 ⑧의 어느 줄 때문일까?
4. 다음 가슴 날 기록을 메모 **맨 아래에** `가슴`으로 시작해 추가하면, ⑥의 가슴 종목들은 무엇과 무엇을 비교하게 될까?